# $K$-armed bandit problem

### ***Summer School - June, 29 2026***

A statistician walks into a casino and is faced with several slot machines. Some of the machines pay out frequently when played, while others pay out much less often.  She has a certain number of plays available and aims to maximise her winnings. What are the best strategies? 


There is $K \in \mathbb{N}^\star$ slot machines in the casino. The learner has $T\in \mathbb{N}^\star$ moves to play and aims to maximise her winnings.

For $t = 1,\dots, T$

$\qquad \bullet$ Pick a slot machine $A_t \in \{1,\dots, K\}$

$\qquad \bullet$ Receive reward $Y_t$ with $Y_t \, | \, A_t = k \sim \nu_k$

We first simulate a completly random strategy (linear regret). 

In [ ]:
# import libraries
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython import display
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100

#  Choice of slot machine number
K = 3

# Generation of Bernoulli distribution parameters 
# mu = np.random.uniform(0,1,K)

# Or manual definition of the distribution parameters
mu = np.array([0.5,0.2,0.6])

# Horizon
T = 500

# Simulation of the gains (full-information setting)
# Matrix of size K x T that will contain the winnings from each machine 
Y_fullinfo = np.random.binomial(1, mu, size=(T, 3)).T

In [ ]:
def play_random(Y_fullinfo):
    """
    Y_fullinfo : Matrix K x T containing all the gains (full information settings)
    """
    # Vector of size T that will contain the numbers of the machines played 
    K, T = Y_fullinfo.shape
    At = -1*np.ones(T)
    G = np.zeros([K,T])
    for t in range(T):
        # The learner chooses the slot machine at random. 
        at = math.floor(random.uniform(0,K))
        # a_t is stored in the vector At
        At[t] = at 
        # She received the reward gt : il est tiré selon la loi de Bernoulli de paramètre p[a_t]
        # For each slot machine
        for k in range(K):
            # if k is played, then the cumulative reward associated with machine k is updated
            if k == at:
                G[k,t] = G[k,t-1] + Y_fullinfo[k,t]
            # if k is not played, then the cumulative reward associated with machine k stays the same
            else:
                G[k,t] = G[k,t-1] 
    return At, G

At_rd, Y_rd = play_random(Y_fullinfo)

## I. Explore then commit

The learner explores in a deterministic manner during $K \times T_{\mathrm{explore}}$ moves : she picks each slot machine $T_{\mathrm{explore}}$ times (*Explore* stage).
Then, she estimates $p_k$ for each $k$ (empirical mean).
The rest of the time, it will play the machine with the highest empirical mean, i.e. the one that earned it the most during the exploration phase (*Commit* stage).

In [ ]:
# Explore then commit 

def play_etc(Y_fullinfo, prop_explore = -1):
    
    """
    Y_fullinfo : Matrix K x T containing all the gains (full information settings)
    prop_explore : proportion of time dedicated to exploration 
    """
    
    # Vector of size T that will contain the numbers of the machines played 
    K, T = Y_fullinfo.shape

    if prop_explore <0:
        prop_explore = (T**(2/3))/T

    # Matrix of size K x T  that will contain the winnings from each machine 
    Y_etc = np.zeros([K, T])

    # Vector of size T that will contain the numbers of the machines played 
    At_etc  = -1*np.ones(T)

    # Vector of size K that will contain the number of times each machine has been played 
    N = np.zeros([K])
    
    # m : number of time each slot machine is played
    m = math.floor(prop_explore*T/(K))

    # Explore then commit algorithm: 
    for t in range(T):
        # explore
        if t<(K*m):
            at = math.floor(t/m)

        #commit 
        else:
            at = np.argmax(Y_etc[:,K*m-1])
        At_etc[t] = at 
        for k in range(K):
            if k == at:
                Y_etc[k,t] = Y_etc[k,t-1] + Y_fullinfo[k,t]
                N[k] = N[k] + 1
            else:
                Y_etc[k,t] = Y_etc[k,t-1] 
    return At_etc, Y_etc


At_etc, Y_etc = play_etc(Y_fullinfo)

**Sensitivity to exploration proportion factor:**
<style>
o { color: Orange }
</style>

<span style="font-size: 24px;"><o> ☞ </o> </span> Pick a proportion `prop` for exploration and tell us your cumulative reward!

In [ ]:
np.random.seed(0)
Y_fullinfo = np.random.binomial(1, mu, size=(T, 3)).T
_, Y_etc = play_etc(Y_fullinfo)
Y_etc.sum(axis = 0)[T-1]

## II. Upper confidence bound

The *Upper confidence bound* (UCB, Auer et al., 2002) algorithm lies on the principle of optimism in the face of uncertainty.

For each $k=1,\dots,K$, the learner approximates $p_k$ (empirical mean - we denote $\widehat{p}_{k,t}$ the estimation of $p_k$ at $t$) and builds a confidence interval $\alpha_{k,t}$ arround $p_k$ such that, with high probability,  $p_k \in [\widehat{p}_{k,t} - \alpha_{k,t} ,\widehat{p}_{k,t} + \alpha_{k,t} ]$.
This is where optimism comes in: she imagines that all the machines are as profitable as is very likely possible; in other words, she imagines that for all slot machines, $p_k = \widehat{p}_{k,t} + \alpha_{k,t}$ and therefore chooses
$A_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}}\{ \widehat{p}_{k,t} + \alpha_{k,t} \}$. 

The challenge now lies in choosing the $\alpha_{k,t}$. The theory (Hoeffding inequality) gives the explicit expression:
$$A_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}} \Bigg\{ \widehat{p}_{k,t} + \alpha \sqrt{\frac{ \log t}{N_{k,t}}} \Bigg\} $$
with $N_{k,t}$ the number of times the statistician plays machine $k$ after move $t$.

In [ ]:
def play_ucb(Y_fullinfo, alpha = np.sqrt(2)):
    """
    Y_fullinfo : Matrix K x T containing all the gains (full information settings)
    alpha : exploration constant (theroy claims a=1 for Bernoulli distributions) 
    """
    K, T = Y_fullinfo.shape

    Y_ucb = np.zeros([K, T])
    At_ucb = -1*np.ones(T)
    ucb = np.zeros([K, T])
    N = np.zeros([K,T])

    for t in range(T):
        if t<K:
            at = t
        else:
            at = np.argmax(ucb[:,t-1])
        At_ucb[t] = at 
        for k in range(K):
            if k == at:
                Y_ucb[k,t] = Y_ucb[k,t-1] + Y_fullinfo[k,t]
                if t == 1:
                    N[k,t] = 1*(k == at)
                else:
                    N[k,t] = N[k,t-1] + 1
            else:
                Y_ucb[k,t] = Y_ucb[k,t-1] 
                N[k,t] = N[k,t-1] 
            ucb[k,t] = Y_ucb[k,t]/max(N[k, t],1) + alpha*math.sqrt(math.log(t+1)/max(N[k,t],1))
    return At_ucb, Y_ucb, N

At_ucb, Y_ucb, N = play_ucb(Y_fullinfo)

def PlotUCB(t): 
    ylm = -0.7
    ylM = 2
    plt.cla()   
    plt.clf()
    plt.ylim((ylm,ylM))
    plt.xlim((-0.4,K-0.6))
    plt.axhline(y=0, color = 'red', linestyle = 'dotted')
    plt.axhline(y=1, color = 'red', linestyle = 'dotted')

    at = At_ucb[t]
    for k in range(K):
        if k == at:
            c = "#01A9B1"
        else:
            c = '#22509D'
        plt.axvline(x= k , 
                ymin= (Y_ucb[k,t]/max(N[k, t],1) - (math.sqrt(math.log(max(t+1,2))/max(N[k,t],1)))-ylm )/(ylM-ylm),
                ymax= (Y_ucb[k,t]/max(N[k, t],1) + (math.sqrt(math.log(max(t+1,2))/max(N[k,t],1))) -ylm  )/(ylM-ylm) ,
                color=c, 
                marker = 'x', markersize=7)
        plt.plot(k,(Y_ucb[k,t]/max(N[k, t],1)),c,marker='o') 
        plt.plot(k,mu[k],'k',marker='*', markersize=7) 
    plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    bottom=False,      # ticks along the bottom edge are off
    top=False,         # ticks along the top edge are off
    labelbottom=False)


f, ax1 = plt.subplots(1,1)
anim_created = animation.FuncAnimation(f,PlotUCB, frames=T, interval=300)
video = anim_created.to_jshtml()
html = display.HTML(video)
display.display(html) 
plt.close()


In the previous animation, each segment corresponds to the confidence interval $[\widehat{p}_{k,t} - \alpha_{k,t} ,\widehat{p}_{k,t} + \alpha_{k,t} ]$ for each slot machine $k = 1,\dots,K$, the black stars correspond to the true values $p_k$ and the circles to the estimated values $\widehat{p}_{k,t}$. The machine being played (the one with the highest confidence interval) lights up in light blue, as in the first animations.  

**Sensitivity to exploration factor $\alpha$:**
$$A_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}} \Bigg\{ \widehat{p}_{k,t} + \alpha\sqrt{\frac{\log t}{N_{k,t}}} \Bigg\} \,. $$

<style>
o { color: Orange }
</style>

<span style="font-size: 24px;"><o> ☞ </o> </span> Pick an exploration factor `alpha` tell us your cumulative reward! 

In [ ]:
np.random.seed(0)
Y_fullinfo = np.random.binomial(1, mu, size=(T, 3)).T
_, Y_ucb, _ = play_ucb(Y_fullinfo)
Y_ucb.sum(axis = 0)[T-1]

## III. Thompson Sampling

The Thompson sampling algorithm (Thompson, 1933; Chapelle and Li, 2011; Scott, 2010) is a Bayesian-sampling-based strategy.

For each $k=1,\dots,K$, the learner gives a prior distributions for $k$-machine rewawrds: a Beta distribution with parameters ($a_k$=1,$b_k$=1). 
For $t=1,\dots,T$, she will simulates the machine rewards by randomly drawing according to the prior distributions and then proceeds as if the simulated rewards were real: she chooses the machine with the highest ones. Then, based on the actual observed reward, she updates the distribution a posteriori: for the chosen machine $k$, $a_k$= $a_k$+1 if the reward is $1$; $b_k=b_k+1$ otherwise.

In [ ]:
# Thompson sampling
from scipy.stats import beta

def play_ts(Y_fullinfo):
    
    """
    Y_fullinfo : Matrix K x T containing all the gains (full information settings)
    """
    
    # Vector of size T that will contain the numbers of the machines played 
    K, T = Y_fullinfo.shape
    # Matrix of size K x T  that will contain the winnings from each machine 
    Y_ts = np.zeros([K, T])

    # Vector of size T that will contain the numbers of the machines played 
    At_ts = -1*np.ones(T)

    # Vector of size K that will contain the number of times each machine has been played 
    N = np.zeros([K])

    # A and B (Beta distribution parameters)
    A = np.ones([K])
    B = np.ones([K])
    Y_sim = -1*np.ones([K])

    # Store parameters 
    A_ts = np.zeros([K, T])
    B_ts = np.zeros([K, T])


    # Explore then commit algorithm: 
    for t in range(T):
        A_ts[:,t] = A
        B_ts[:,t] = B 
        # simulate rewards
        Y_sim = np.random.beta(A,B)

        at = np.argmax(Y_sim)
        At_ts[t] = at 
        Yt = Y_fullinfo[at,t]

        if Yt == 1:
            A[at] += 1
        else:
            B[at] += 1


        for k in range(K):
            if k == at:
                Y_ts[k,t] = Y_ts[k,t-1] + Yt
                N[k] = N[k] + 1
            else:
                Y_ts[k,t] = Y_ts[k,t-1] 
    return At_ts, Y_ts, A_ts, B_ts

At_ts, Y_ts, A_ts, B_ts = play_ts(Y_fullinfo)

def PlotTS(t): 
    ylm = 0
    ylM = 8
    plt.cla()   
    plt.clf()
    plt.ylim((ylm,ylM))
    plt.xlim((0,1))
    at = At_ts[t]
    ln = ['solid','dotted','dashed','dashdot']
    for k in range(K):
        if k == at:
            c = "#01A9B1"
        else:
            c = '#22509D'
        a = A_ts[k, t] 
        b = B_ts[k, t]
        x = np.linspace(0, 1, 1000)
        plt.plot(x, beta(a,b).pdf(x), color=c, linestyle= ln[k%len(ln)])
    plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    labelbottom=False)


f, ax1 = plt.subplots(1,1)
anim_created = animation.FuncAnimation(f,PlotTS, frames=T, interval=300)
video = anim_created.to_jshtml()
html = display.HTML(video)
display.display(html) 
plt.close()


In the previous animation, for each of the slot machines $k = 1,\dots,K$, with different line styles, the prior Beta distribution is plotted. The distribution of the machine being played (the one with the highest confidence interval) lights up in light blue. 

## IV. Comparison of strategies (Random, Explore then commit, Upper confidence bound, Thompson sampling)

In [ ]:
alpha_opt = np.sqrt(2)
prop_opt = T**(2/3)/T

palette = {
    "Random": "#223F6A",
    "ETC": "#D5B813",
    "UCB": "#326DC0",
    "TS": "#098D40"
}

n = 500

algos = {
    "Random": lambda G: play_random(G)[:2],
    "ETC":    lambda G: play_etc(G, prop_opt)[:2],
    "UCB":    lambda G: play_ucb(G, alpha_opt)[:2],
    "TS":     lambda G: play_ts(G)[:2],
}

results = {name: np.zeros((n, T)) for name in algos}

# Simulations
for i in range(n):
    Y_fullinfo = np.random.binomial(1, mu, size=(T, K)).T

    for name, algo in algos.items():
        _, G = algo(Y_fullinfo)
        results[name][i] = np.sum(G, axis=0)

x = np.arange(T)
means = {k: v.mean(axis=0) for k, v in results.items()}
stds  = {k: v.std(axis=0)  for k, v in results.items()}

fig, ax = plt.subplots()

for name in algos:
    ax.plot(x, means[name], label=name, color=palette[name])
    ax.fill_between(
        x,
        means[name] - stds[name],
        means[name] + stds[name],
        color=palette[name],
        alpha=0.2
    )

ax.set_xlabel("Iterations")

ax.set_ylabel("Cumulative reward")
ax.legend()
plt.show()

y_best = np.arange(1, T+1, 1)*max(p)
regret = {k: y_best - v.mean(axis=0) for k, v in results.items()}

fig, ax = plt.subplots()

for name in algos:
    ax.plot(x, regret[name], label=name, color=palette[name])

ax.set_xlabel("Iterations")
ax.set_ylabel("Regret")
ax.legend()
plt.show()